# A/B Test Statistical Analysis Framework

Universal toolkit for evaluating any A/B test experiment. Supports both **binary metrics** (conversion rates, cage rates, click-through rates) and **continuous metrics** (revenue per user, avg documents per shipment).

**How to use:** Update Section 1 with your experiment data and configuration. Everything else runs automatically.

### Variables to update for each experiment:

| Variable | Description | Example |
|----------|-------------|----------|
| `experiment_id` | Your experiment ID | `"492470"` |
| `experiment_name` | Short description of the test | `"Item Description Notification"` |
| `test_duration_days` | Number of days the test ran | `17` |
| `metric_type` | `"binary"` for proportions, `"continuous"` for means | `"binary"` |
| `metric_name` | Name used in all labels and charts | `"Cage Rate"` |
| `lower_is_better` | `True` if lower = good (cage rate, bounce rate), `False` if higher = good (CTR, revenue) | `True` |
| `control_label` | Display name for control group | `"Control (OFF)"` |
| `treatment_label` | Display name for treatment group | `"Treatment (ON)"` |

**If `metric_type = "binary"`:**

| Variable | Description | Example |
|----------|-------------|----------|
| `control_users` | Total observations in control | `640` |
| `control_conversions` | Successes/events in control | `53` |
| `test_users` | Total observations in treatment | `1530` |
| `test_conversions` | Successes/events in treatment | `45` |

**If `metric_type = "continuous"`:**

| Variable | Description | Example |
|----------|-------------|----------|
| `control_n` | Observations in control | `50000` |
| `control_mean` | Mean metric value in control | `1.2345` |
| `control_std` | Std deviation (or `None` for Poisson approx) | `None` |
| `test_n` | Observations in treatment | `48000` |
| `test_mean` | Mean metric value in treatment | `1.3456` |
| `test_std` | Std deviation (or `None` for Poisson approx) | `None` |

---

## 0. Setup & Imports

In [ ]:
import numpy as np
from scipy import stats
from scipy.stats import beta
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
import math
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
np.random.seed(42)

## 1. Experiment Configuration

This is the **only cell you need to update** for each experiment. Fill in your data and the notebook adapts all labels, analysis direction, and recommendations.

### Metric types supported:
- **Binary (proportion):** conversion rate, cage rate, click-through rate, bounce rate, etc.
- **Continuous (mean):** revenue per user, avg docs per shipment, session duration, etc.

In [ ]:
# ============================================================
# >>>  EXPERIMENT METADATA  <<<
# ============================================================

experiment_id = "492470"                              # Experiment ID
experiment_name = "Item Description Notification"     # Short description
test_duration_days = 17                               # Days the test ran

# ============================================================
# >>>  METRIC CONFIGURATION  <<<
# ============================================================

metric_type = "binary"          # "binary" (proportions) or "continuous" (means)
metric_name = "Cage Rate"       # Name of the metric (used in all labels)
lower_is_better = True          # True: lower = good (cage rate, bounce rate)
                                # False: higher = good (conversion rate, CTR, revenue)

# ============================================================
# >>>  GROUP LABELS  <<<
# ============================================================

control_label = "Control (OFF)"    # Label for control group
treatment_label = "Treatment (ON)" # Label for treatment group

# ============================================================
# >>>  DATA INPUT — BINARY METRICS  <<<
# (Fill this section if metric_type = "binary")
# ============================================================

control_users = 640              # Total observations in control
control_conversions = 53         # Successes/events in control

test_users = 1530                # Total observations in treatment
test_conversions = 45            # Successes/events in treatment

# ============================================================
# >>>  DATA INPUT — CONTINUOUS METRICS  <<<
# (Fill this section if metric_type = "continuous")
# ============================================================

control_n = 50000                # Observations in control
control_mean = 1.2345            # Mean metric value in control
control_std = None               # Std dev (set None to use Poisson approx: std = sqrt(mean))

test_n = 48000                   # Observations in treatment
test_mean = 1.3456               # Mean metric value in treatment
test_std = None                  # Std dev (set None to use Poisson approx)

# ============================================================
# >>>  STATISTICAL SETTINGS  <<<
# ============================================================

alpha = 0.05                     # Significance level
desired_power = 0.80             # Target power for sample size planning
n_mc_samples = 100000            # Monte Carlo samples for Bayesian analysis

# ============================================================
print(f'Experiment #{experiment_id}: {experiment_name}')
print(f'Metric: {metric_name} ({metric_type}) | Direction: {"Lower is better" if lower_is_better else "Higher is better"}')
print(f'Duration: {test_duration_days} days | Alpha: {alpha}')
print(f'Groups: {control_label} vs {treatment_label}')

---
## 2. Derived Values & Validation

Auto-computes rates, differences, and validates data integrity.

In [ ]:
# ============================================================
# Compute derived values based on metric type
# ============================================================

if metric_type == 'binary':
    control_rate = control_conversions / control_users
    test_rate = test_conversions / test_users
    total_users = control_users + test_users
    n_control = control_users
    n_test = test_users
    rate_label = f'{metric_name} (%)'
    
elif metric_type == 'continuous':
    control_rate = control_mean
    test_rate = test_mean
    total_users = control_n + test_n
    n_control = control_n
    n_test = test_n
    # Variance: use provided std or Poisson approximation
    control_var = control_std**2 if control_std else control_mean
    test_var = test_std**2 if test_std else test_mean
    rate_label = f'{metric_name}'

else:
    raise ValueError(f'Unknown metric_type: {metric_type}. Use "binary" or "continuous".')

difference = test_rate - control_rate
relative_change = (difference / control_rate) * 100 if control_rate != 0 else 0

# Direction labels
if lower_is_better:
    good_direction = 'decrease'
    bad_direction = 'increase'
else:
    good_direction = 'increase'
    bad_direction = 'decrease'

is_treatment_good = (difference < 0) if lower_is_better else (difference > 0)
is_treatment_bad = not is_treatment_good and difference != 0

print('=' * 60)
print('EXPERIMENT SUMMARY')
print('=' * 60)

if metric_type == 'binary':
    print(f'\n{control_label}: {control_conversions:,} / {control_users:,} = {control_rate*100:.2f}%')
    print(f'{treatment_label}: {test_conversions:,} / {test_users:,} = {test_rate*100:.2f}%')
    print(f'\nAbsolute difference: {difference*100:+.2f} pp')
else:
    print(f'\n{control_label}: {control_rate:.4f} (n={n_control:,})')
    print(f'{treatment_label}: {test_rate:.4f} (n={n_test:,})')
    print(f'\nAbsolute difference: {difference:+.4f}')

print(f'Relative change: {relative_change:+.1f}%')
outcome = 'GOOD' if is_treatment_good else ('BAD' if is_treatment_bad else 'NEUTRAL')
print(f'Direction: Treatment caused a {bad_direction if is_treatment_bad else good_direction} ({outcome})')

---
## 3. Traffic Allocation Check

In [ ]:
print('=' * 60)
print('TRAFFIC ALLOCATION')
print('=' * 60)

control_pct = n_control / total_users * 100
test_pct = n_test / total_users * 100
deviation = abs(50 - control_pct)

print(f'\n{control_label}:  {n_control:,} ({control_pct:.1f}%)')
print(f'{treatment_label}: {n_test:,} ({test_pct:.1f}%)')
print(f'Total:           {total_users:,}')
print(f'Deviation from 50/50: {deviation:.1f} pp')

if deviation <= 2:
    print('\nTraffic allocation looks balanced.')
elif deviation <= 5:
    print('\nSlight imbalance — acceptable but worth noting.')
else:
    print('\nSignificant traffic imbalance detected!')
    print('   This may affect result reliability. Investigate randomization.')

---
## 4. Statistical Significance Test

- **Binary metrics:** Two-proportion z-test
- **Continuous metrics:** Welch's t-test

In [ ]:
print('=' * 60)
print('STATISTICAL SIGNIFICANCE TEST')
print('=' * 60)

if metric_type == 'binary':
    # Two-proportion z-test
    test_name = 'Two-proportion z-test'
    successes = [control_conversions, test_conversions]
    totals = [control_users, test_users]
    z_stat, p_value = proportions_ztest(successes, totals)
    stat_name = 'Z-statistic'
    stat_value = z_stat

elif metric_type == 'continuous':
    # Welch's t-test
    test_name = "Welch's t-test"
    se = np.sqrt(control_var / n_control + test_var / n_test)
    stat_value = difference / se
    # Welch-Satterthwaite degrees of freedom
    num = (control_var / n_control + test_var / n_test) ** 2
    denom = (control_var / n_control)**2 / (n_control - 1) + (test_var / n_test)**2 / (n_test - 1)
    df = num / denom
    p_value = 2 * (1 - stats.t.cdf(abs(stat_value), df))
    stat_name = 'T-statistic'
    z_stat = stat_value  # Store for later use

print(f'\nTest: {test_name}')
print(f'{stat_name}: {stat_value:.4f}')
if metric_type == 'continuous':
    print(f'Degrees of freedom: {df:.0f}')
print(f'P-value: {p_value:.6f}')
print(f'\nSignificant at alpha=0.10: {"YES" if p_value < 0.10 else "NO"}')
print(f'Significant at alpha=0.05: {"YES" if p_value < 0.05 else "NO"}')
print(f'Significant at alpha=0.01: {"YES" if p_value < 0.01 else "NO"}')

---
## 5. Confidence Intervals (95%)

In [ ]:
print('=' * 60)
print('CONFIDENCE INTERVALS (95%)')
print('=' * 60)

if metric_type == 'binary':
    # Wilson score intervals
    control_ci = proportion_confint(control_conversions, control_users, alpha=0.05, method='wilson')
    test_ci = proportion_confint(test_conversions, test_users, alpha=0.05, method='wilson')
    
    print(f'\n{control_label} 95% CI: [{control_ci[0]*100:.2f}%, {control_ci[1]*100:.2f}%]')
    print(f'{treatment_label} 95% CI: [{test_ci[0]*100:.2f}%, {test_ci[1]*100:.2f}%]')
    
    se_diff = np.sqrt(control_rate*(1-control_rate)/n_control + test_rate*(1-test_rate)/n_test)
    diff_ci_lower = difference - 1.96 * se_diff
    diff_ci_upper = difference + 1.96 * se_diff
    print(f'\nDifference 95% CI: [{diff_ci_lower*100:.2f} pp, {diff_ci_upper*100:.2f} pp]')

elif metric_type == 'continuous':
    se_ctrl = np.sqrt(control_var / n_control)
    se_trt = np.sqrt(test_var / n_test)
    control_ci = (control_rate - 1.96*se_ctrl, control_rate + 1.96*se_ctrl)
    test_ci = (test_rate - 1.96*se_trt, test_rate + 1.96*se_trt)
    
    print(f'\n{control_label} 95% CI: [{control_ci[0]:.4f}, {control_ci[1]:.4f}]')
    print(f'{treatment_label} 95% CI: [{test_ci[0]:.4f}, {test_ci[1]:.4f}]')
    
    se_diff = np.sqrt(control_var/n_control + test_var/n_test)
    diff_ci_lower = difference - 1.96 * se_diff
    diff_ci_upper = difference + 1.96 * se_diff
    print(f'\nDifference 95% CI: [{diff_ci_lower:.4f}, {diff_ci_upper:.4f}]')

ci_contains_zero = diff_ci_lower <= 0 <= diff_ci_upper
print(f'\nCI contains zero: {"YES — difference may not be real" if ci_contains_zero else "NO — difference is likely real"}')

---
## 6. Effect Size

In [ ]:
print('=' * 60)
print('EFFECT SIZE')
print('=' * 60)

if metric_type == 'binary':
    # Cohen's h for proportions
    effect_size = 2 * (np.arcsin(np.sqrt(test_rate)) - np.arcsin(np.sqrt(control_rate)))
    effect_name = "Cohen's h"
    small_thresh, med_thresh = 0.2, 0.5
else:
    # Cohen's d for continuous
    pooled_std = np.sqrt((control_var + test_var) / 2)
    effect_size = difference / pooled_std if pooled_std > 0 else 0
    effect_name = "Cohen's d"
    small_thresh, med_thresh = 0.2, 0.5

print(f'\n{effect_name}: {effect_size:.4f}')

abs_effect = abs(effect_size)
if abs_effect < small_thresh:
    effect_magnitude = f'Small (|{effect_name}| < {small_thresh})'
elif abs_effect < med_thresh:
    effect_magnitude = f'Medium ({small_thresh} <= |{effect_name}| < {med_thresh})'
else:
    effect_magnitude = f'Large (|{effect_name}| >= {med_thresh})'

print(f'Magnitude: {effect_magnitude}')

---
## 7. Power Analysis

In [ ]:
print('=' * 60)
print('POWER ANALYSIS')
print('=' * 60)

if metric_type == 'binary':
    def calculate_power_binary(n1, n2, p1, p2, alpha=0.05):
        pooled_p = (n1 * p1 + n2 * p2) / (n1 + n2)
        se_null = np.sqrt(pooled_p * (1 - pooled_p) * (1/n1 + 1/n2))
        se_alt = np.sqrt(p1*(1-p1)/n1 + p2*(1-p2)/n2)
        z_crit = stats.norm.ppf(1 - alpha/2)
        z_beta = (abs(p1 - p2) - z_crit * se_null) / se_alt
        return max(0, min(1, stats.norm.cdf(z_beta)))
    
    current_power = calculate_power_binary(n_control, n_test, control_rate, test_rate, alpha)
    
    # MDE
    z_a = stats.norm.ppf(1 - alpha/2)
    z_b = stats.norm.ppf(desired_power)
    mde = (z_a + z_b) * np.sqrt(control_rate*(1-control_rate) * (1/n_control + 1/n_test))
    mde_display = f'{mde*100:.2f} pp'
    observed_display = f'{abs(difference)*100:.2f} pp'

elif metric_type == 'continuous':
    se_null = np.sqrt((control_var + test_var) / 2 * (1/n_control + 1/n_test))
    se_alt = np.sqrt(control_var/n_control + test_var/n_test)
    z_crit = stats.norm.ppf(1 - alpha/2)
    z_power = (abs(difference) - z_crit * se_null) / se_alt
    current_power = max(0, min(1, stats.norm.cdf(z_power)))
    
    z_a = stats.norm.ppf(1 - alpha/2)
    z_b = stats.norm.ppf(desired_power)
    mde = (z_a + z_b) * se_alt
    mde_display = f'{mde:.4f}'
    observed_display = f'{abs(difference):.4f}'

print(f'\nCurrent statistical power: {current_power:.3f} ({current_power*100:.1f}%)')

if current_power >= 0.80:
    print('Adequate power (>= 80%) — results are reliable.')
elif current_power >= 0.50:
    print('Moderate power — directionally informative but not conclusive.')
else:
    print('Low power — test is underpowered. Consider extending.')

print(f'\nMinimum Detectable Effect (at {desired_power*100:.0f}% power): {mde_display}')
print(f'Observed effect: {observed_display}')
if abs(difference) >= mde if metric_type == 'continuous' else abs(difference) >= mde:
    print('Observed effect exceeds MDE.')
else:
    print('Observed effect is below MDE — more data may be needed.')

---
## 8. Bayesian Analysis

- **Binary metrics:** Beta-binomial conjugate model with Beta(1,1) prior
- **Continuous metrics:** Normal approximation via Monte Carlo

In [ ]:
print('=' * 60)
print('BAYESIAN ANALYSIS')
print('=' * 60)

np.random.seed(42)

if metric_type == 'binary':
    # Beta-binomial conjugate model
    a_ctrl = 1 + control_conversions
    b_ctrl = 1 + control_users - control_conversions
    a_trt = 1 + test_conversions
    b_trt = 1 + test_users - test_conversions
    
    ctrl_samples = np.random.beta(a_ctrl, b_ctrl, n_mc_samples)
    trt_samples = np.random.beta(a_trt, b_trt, n_mc_samples)
    
    post_ctrl = a_ctrl / (a_ctrl + b_ctrl)
    post_trt = a_trt / (a_trt + b_trt)

elif metric_type == 'continuous':
    # Normal approximation for posterior
    ctrl_se = np.sqrt(control_var / n_control)
    trt_se = np.sqrt(test_var / n_test)
    
    ctrl_samples = np.random.normal(control_rate, ctrl_se, n_mc_samples)
    trt_samples = np.random.normal(test_rate, trt_se, n_mc_samples)
    
    post_ctrl = control_rate
    post_trt = test_rate

prob_trt_higher = np.mean(trt_samples > ctrl_samples)
prob_trt_lower = np.mean(trt_samples < ctrl_samples)

print(f'\nPosterior mean ({control_label}):  {post_ctrl:.4f}')
print(f'Posterior mean ({treatment_label}): {post_trt:.4f}')
print(f'\nP(Treatment > Control): {prob_trt_higher:.4f} ({prob_trt_higher*100:.1f}%)')
print(f'P(Treatment < Control): {prob_trt_lower:.4f} ({prob_trt_lower*100:.1f}%)')

# Direction-aware interpretation
if lower_is_better:
    prob_treatment_good = prob_trt_lower
    prob_treatment_bad = prob_trt_higher
else:
    prob_treatment_good = prob_trt_higher
    prob_treatment_bad = prob_trt_lower

print(f'\nP(Treatment is beneficial — {good_direction}): {prob_treatment_good*100:.1f}%')
print(f'P(Treatment is harmful — {bad_direction}):    {prob_treatment_bad*100:.1f}%')

if prob_treatment_bad > 0.95:
    print('\nStrong Bayesian evidence: treatment is HARMFUL.')
elif prob_treatment_good > 0.95:
    print('\nStrong Bayesian evidence: treatment is BENEFICIAL.')
elif prob_treatment_bad > 0.80:
    print('\nModerate evidence treatment is harmful.')
elif prob_treatment_good > 0.80:
    print('\nModerate evidence treatment is beneficial.')
else:
    print('\nBayesian analysis is inconclusive — continue testing.')

---
## 9. Sample Size Planning

In [ ]:
print('=' * 60)
print('SAMPLE SIZE PLANNING FOR FUTURE TESTS')
print('=' * 60)

daily_traffic = total_users / test_duration_days
print(f'\nCurrent daily traffic: ~{daily_traffic:.0f} observations/day')

if metric_type == 'binary':
    print(f'Baseline {metric_name}: {control_rate*100:.2f}%\n')
    
    def sample_size_binary(p1, p2, alpha=0.05, power=0.8):
        p_pool = (p1 + p2) / 2
        es = abs(p1 - p2)
        if es == 0: return float('inf')
        za = stats.norm.ppf(1 - alpha/2)
        zb = stats.norm.ppf(power)
        return math.ceil((za + zb)**2 * p_pool * (1-p_pool) / es**2)
    
    scenarios = [
        ('5% relative change', control_rate * 1.05),
        ('10% relative change', control_rate * 1.10),
        ('20% relative change', control_rate * 1.20),
        ('50% relative change', control_rate * 1.50),
        ('Current observed effect', test_rate),
    ]
    
    print(f'{"Scenario":<30} {"N/group":>10} {"Total N":>10} {"Days":>8}')
    print('-' * 63)
    for name, p2 in scenarios:
        if abs(p2 - control_rate) > 0:
            n = sample_size_binary(control_rate, p2, alpha, desired_power)
            d = (n * 2) / daily_traffic if daily_traffic > 0 else float('inf')
            print(f'{name:<30} {n:>10,} {n*2:>10,} {d:>8.0f}')

elif metric_type == 'continuous':
    print(f'Baseline {metric_name}: {control_rate:.4f}\n')
    
    pooled_var = (control_var + test_var) / 2
    
    def sample_size_continuous(effect, var, alpha=0.05, power=0.8):
        if effect == 0: return float('inf')
        za = stats.norm.ppf(1 - alpha/2)
        zb = stats.norm.ppf(power)
        return math.ceil(2 * (za + zb)**2 * var / effect**2)
    
    scenarios = [
        ('1% relative change', abs(control_rate * 0.01)),
        ('5% relative change', abs(control_rate * 0.05)),
        ('10% relative change', abs(control_rate * 0.10)),
        ('Current observed effect', abs(difference)),
    ]
    
    print(f'{"Scenario":<30} {"N/group":>10} {"Total N":>10} {"Days":>8}')
    print('-' * 63)
    for name, eff in scenarios:
        if eff > 0:
            n = sample_size_continuous(eff, pooled_var, alpha, desired_power)
            d = (n * 2) / daily_traffic if daily_traffic > 0 else float('inf')
            print(f'{name:<30} {n:>10,} {n*2:>10,} {d:>8.0f}')

---
## 10. Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f'A/B Test Results — #{experiment_id}: {experiment_name}', fontsize=16, fontweight='bold')

# ---- Plot 1: Rates with CI ----
ax1 = axes[0, 0]
groups = [control_label, treatment_label]

if metric_type == 'binary':
    rates_plot = [control_rate * 100, test_rate * 100]
    ci_lo = [control_ci[0] * 100, test_ci[0] * 100]
    ci_hi = [control_ci[1] * 100, test_ci[1] * 100]
    fmt = lambda v: f'{v:.2f}%'
    ylabel = f'{metric_name} (%)'
else:
    rates_plot = [control_rate, test_rate]
    ci_lo = [control_ci[0], test_ci[0]]
    ci_hi = [control_ci[1], test_ci[1]]
    fmt = lambda v: f'{v:.4f}'
    ylabel = metric_name

errors = [[r - l for r, l in zip(rates_plot, ci_lo)],
          [u - r for r, u in zip(rates_plot, ci_hi)]]
colors = ['#4CAF50', '#f44336']
bars = ax1.bar(groups, rates_plot, yerr=errors, capsize=8, color=colors, alpha=0.8, edgecolor='black')
ax1.set_ylabel(ylabel, fontweight='bold')
ax1.set_title(f'{metric_name} with 95% CI', fontweight='bold')
for bar, rate in zip(bars, rates_plot):
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + max(errors[1])*0.1,
             fmt(rate), ha='center', va='bottom', fontweight='bold')

# ---- Plot 2: Posterior distributions ----
ax2 = axes[0, 1]
if metric_type == 'binary':
    x = np.linspace(0, max(control_rate, test_rate) * 2.5, 1000)
    x = x[x > 0]
    ctrl_pdf = beta.pdf(x, a_ctrl, b_ctrl)
    trt_pdf = beta.pdf(x, a_trt, b_trt)
    ax2.plot(x * 100, ctrl_pdf, color='#4CAF50', linewidth=2.5, label=control_label, alpha=0.9)
    ax2.plot(x * 100, trt_pdf, color='#f44336', linewidth=2.5, label=treatment_label, alpha=0.9)
    ax2.fill_between(x * 100, ctrl_pdf, alpha=0.2, color='#4CAF50')
    ax2.fill_between(x * 100, trt_pdf, alpha=0.2, color='#f44336')
    ax2.set_xlabel(f'{metric_name} (%)', fontweight='bold')
else:
    ax2.hist(ctrl_samples, bins=60, alpha=0.5, color='#4CAF50', label=control_label, density=True)
    ax2.hist(trt_samples, bins=60, alpha=0.5, color='#f44336', label=treatment_label, density=True)
    ax2.set_xlabel(metric_name, fontweight='bold')

ax2.set_ylabel('Probability Density', fontweight='bold')
ax2.set_title('Bayesian Posterior Distributions', fontweight='bold')
ax2.legend(fontsize=10)

# ---- Plot 3: Effect breakdown ----
ax3 = axes[1, 0]
if metric_type == 'binary':
    vals = [control_rate*100, difference*100, test_rate*100]
    labels_3 = [control_label, 'Difference (pp)', treatment_label]
    fmt3 = [f'{vals[0]:.2f}%', f'{vals[1]:+.2f} pp', f'{vals[2]:.2f}%']
else:
    vals = [control_rate, difference, test_rate]
    labels_3 = [control_label, 'Difference', treatment_label]
    fmt3 = [f'{vals[0]:.4f}', f'{vals[1]:+.4f}', f'{vals[2]:.4f}']

bar_colors = ['#4CAF50', '#FF9800' if is_treatment_bad else '#2196F3', '#f44336']
ax3.bar(labels_3, vals, color=bar_colors, alpha=0.8, edgecolor='black')
for i, (l, v, f) in enumerate(zip(labels_3, vals, fmt3)):
    ax3.text(i, v + abs(max(vals))*0.02, f, ha='center', va='bottom', fontweight='bold', fontsize=9)
ax3.set_ylabel(ylabel, fontweight='bold')
ax3.set_title('Effect Breakdown', fontweight='bold')
ax3.axhline(y=0, color='black', linewidth=0.5)

# ---- Plot 4: MC difference distribution ----
ax4 = axes[1, 1]
diff_samples = trt_samples - ctrl_samples
if metric_type == 'binary':
    diff_samples = diff_samples * 100
    xlabel = f'Treatment - Control (pp)'
else:
    xlabel = f'Treatment - Control'

ax4.hist(diff_samples, bins=80, color='#9C27B0', alpha=0.7, edgecolor='white', density=True)
ax4.axvline(x=0, color='red', linestyle='--', linewidth=2, label='No Effect')
ax4.axvline(x=np.mean(diff_samples), color='blue', linestyle='-', linewidth=2,
            label=f'Mean: {np.mean(diff_samples):.3f}')
ax4.set_xlabel(xlabel, fontweight='bold')
ax4.set_ylabel('Density', fontweight='bold')
ax4.set_title('Monte Carlo: Difference Distribution', fontweight='bold')
ax4.legend(fontsize=9)

plt.tight_layout()
plt.savefig('ab_test_results.png', dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print('\nChart saved: ab_test_results.png')

---
## 11. Final Recommendations

In [ ]:
print('=' * 60)
print(f'FINAL RECOMMENDATIONS — {metric_name}')
print('=' * 60)

print(f'\n--- Summary ---')
print(f'Experiment: #{experiment_id} — {experiment_name}')

if metric_type == 'binary':
    print(f'{control_label}: {control_rate*100:.2f}%  |  {treatment_label}: {test_rate*100:.2f}%')
    print(f'Difference: {difference*100:+.2f} pp ({relative_change:+.1f}% relative)')
else:
    print(f'{control_label}: {control_rate:.4f}  |  {treatment_label}: {test_rate:.4f}')
    print(f'Difference: {difference:+.4f} ({relative_change:+.1f}% relative)')

print(f'P-value: {p_value:.4f}  |  Power: {current_power*100:.1f}%')
print(f'Bayesian P(harmful): {prob_treatment_bad*100:.1f}%  |  P(beneficial): {prob_treatment_good*100:.1f}%')

print(f'\n--- Decision ---')

if is_treatment_bad and p_value < alpha and prob_treatment_bad > 0.95:
    print('STOP TEST — TREATMENT IS HARMFUL')
    print('   Both frequentist and Bayesian analyses confirm harm.')
    print('   Route 100% traffic to control immediately.')
    print('   Investigate root cause before testing new variants.')

elif is_treatment_bad and prob_treatment_bad > 0.90:
    print('LIKELY HARMFUL — Consider Stopping')
    print(f'   P-value ({p_value:.4f}) may not reach significance, but')
    print(f'   Bayesian probability of harm is {prob_treatment_bad*100:.1f}%.')
    print('   Risk outweighs potential benefit. Recommend: revert to control.')

elif is_treatment_good and p_value < alpha and prob_treatment_good > 0.95:
    print('SHIP TREATMENT — SIGNIFICANT IMPROVEMENT')
    print('   Both frequentist and Bayesian analyses confirm benefit.')
    print('   Roll out treatment to 100% of traffic.')

elif is_treatment_good and prob_treatment_good > 0.80:
    print('PROMISING — Continue Testing')
    print('   Directional evidence is positive but not yet conclusive.')
    print('   Continue test to reach statistical significance.')

elif current_power < 0.50:
    print('UNDERPOWERED — Extend Test')
    print('   Insufficient data to draw conclusions.')
    print('   Extend test duration to accumulate more observations.')

else:
    print('NO CLEAR WINNER — Continue or Accept Null')
    print('   No significant difference detected.')
    print('   Either continue testing or accept no meaningful effect.')

print(f'\n--- Next Steps ---')
print('1. Check for confounding variables in your domain')
print('2. Investigate segment-level effects (geography, user type, etc.)')
print('3. Review secondary/guardrail metrics')
print('4. Document findings and share with stakeholders')
print('\n' + '=' * 60)

---
## Appendix: Quick Duration Planner

Standalone calculator — estimate how long an experiment needs to run before you start it.

In [ ]:
# ============================================================
# >>>  PRE-EXPERIMENT PLANNING  <<<
# ============================================================

plan_metric_type = 'binary'      # 'binary' or 'continuous'
plan_baseline = 0.045            # Expected baseline (e.g., 4.5% cage rate, or 1.23 avg docs)
plan_mde_relative = 0.10         # Relative change to detect (e.g., 10% = 0.10)
plan_daily_traffic = 128         # Expected daily traffic across both groups
plan_power = 0.80
plan_alpha = 0.05

print('EXPERIMENT DURATION PLANNER')
print('=' * 60)

if plan_metric_type == 'binary':
    plan_mde_abs = plan_baseline * plan_mde_relative
    p2 = plan_baseline + plan_mde_abs
    p_pool = (plan_baseline + p2) / 2
    za = stats.norm.ppf(1 - plan_alpha/2)
    zb = stats.norm.ppf(plan_power)
    n_per = math.ceil((za + zb)**2 * p_pool * (1-p_pool) / plan_mde_abs**2)
    print(f'Baseline rate: {plan_baseline*100:.1f}%')
    print(f'MDE: {plan_mde_abs*100:.2f} pp ({plan_mde_relative*100:.0f}% relative)')
else:
    plan_mde_abs = plan_baseline * plan_mde_relative
    plan_var = plan_baseline  # Poisson approximation
    za = stats.norm.ppf(1 - plan_alpha/2)
    zb = stats.norm.ppf(plan_power)
    n_per = math.ceil(2 * (za + zb)**2 * plan_var / plan_mde_abs**2)
    print(f'Baseline mean: {plan_baseline:.4f}')
    print(f'MDE: {plan_mde_abs:.4f} ({plan_mde_relative*100:.0f}% relative)')

total_needed = n_per * 2
days = total_needed / plan_daily_traffic

print(f'Daily traffic: {plan_daily_traffic} observations')
print(f'Power: {plan_power*100:.0f}%  |  Alpha: {plan_alpha}')
print(f'\nRequired per group: {n_per:,}')
print(f'Total required: {total_needed:,}')
print(f'Estimated duration: {days:.0f} days ({days/7:.1f} weeks)')